In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# =========================
# Dirty Customers Dataset
# =========================

customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

# =========================
# Dirty Orders Dataset
# =========================

orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

# =========================
# Convert date column
# =========================

orders = orders.withColumn("order_date", to_date(col("order_date")))

# =========================
# Tasks for Students
# =========================




In [0]:
#clean_data
customers_clean = customers.filter(col("name").isNotNull()) \
                           .withColumn("name", trim(col("name")))

orders_clean = orders.filter(col("amount").isNotNull()) \
                     .withColumn("amount", abs(col("amount")))

In [0]:
#validate data
invalid_orders = orders_clean.join(customers_clean, "customer_id", "left_anti")
print(f"Invalid Orphan Orders found: {invalid_orders.count()}")

In [0]:
final_df = orders_clean.join(customers_clean, "customer_id", "inner")

In [0]:
#transformations
customer_metrics = final_df.groupBy("customer_id", "name", "city") \
    .agg(
        sum("amount").alias("total_spent"),
        count("order_id").alias("order_count")
    )

In [0]:
#apply window functions
from pyspark.sql.window import Window
window_spec = Window.orderBy(desc("total_spent"))
ranked_customers = customer_metrics.withColumn("rank", rank().over(window_spec))

ranked_customers.show()

In [0]:
display(ranked_customers)

print("Phase 6 Pipeline completed successfully")


In [0]:
#top1 customer per city
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

window_city = Window.partitionBy("city").orderBy(desc("total_spent"))
top_per_city = customer_metrics.withColumn("city_rank", rank().over(window_city)).filter(col("city_rank") == 1)

In [0]:
#total_sales
window_run = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
running_total_df = final_df.withColumn("running_total", sum("amount").over(window_run))

In [0]:
#data_analysis
from pyspark.sql.functions import month

monthly_sales = final_df.withColumn("month", month("order_date")) \
                        .groupBy("month").agg(sum("amount").alias("monthly_revenue"))

In [0]:
#  Extract Month & Monthly Aggregation
from pyspark.sql.functions import month, sum

# Extract month and calculate total revenue
monthly_trends = final_df.withColumn("month_num", month("order_date")) \
    .groupBy("month_num") \
    .agg(sum("amount").alias("monthly_revenue")) \
    .orderBy("month_num")

monthly_trends.show()

In [0]:
# Calculate Difference Between Dates
from pyspark.sql.functions import datediff, current_date

# How many days has it been since the order was placed?
date_diff_df = final_df.withColumn("days_since_order", datediff(current_date(), col("order_date")))

In [0]:
#Trend Analysis (Month-over-Month)
from pyspark.sql.window import Window
from pyspark.sql.functions import lag

window_month = Window.orderBy("month_num")
trend_df = monthly_trends.withColumn("prev_month_revenue", lag("monthly_revenue").over(window_month)) \
    .withColumn("growth", col("monthly_revenue") - col("prev_month_revenue"))

In [0]:
#data ingestion
customers.printSchema()
orders.printSchema()

In [0]:
#master join
# Inner join ensures only validated data moves to the reporting layer
final_df = orders_clean.join(customers_clean, "customer_id", "inner")

In [0]:
#final report
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

window_spec = Window.partitionBy("city").orderBy(desc("total_spent"))
final_report = customer_metrics.withColumn("rank_in_city", rank().over(window_spec))